# Phân loại ảnh thời trang bằng CNN — Notebook Thực nghiệm nâng cao

Notebook này **kế thừa** notebook gốc `Fashion_MNIST_CNN.ipynb` (giữ nguyên, không chỉnh sửa) — copy toàn bộ phần khung sườn (mục 1-10 bên dưới), sau đó bổ sung các thực nghiệm sâu hơn để phục vụ báo cáo (mục 11-14), theo checklist trong `REPORT_GUIDE.md`.

> Chạy tuần tự từ trên xuống dưới (Runtime > Run all). Các mục 11-14 phụ thuộc vào các biến đã tạo ở mục 1-10 (`cnn_model`, `mlp_model`, `test_set`, `train_idx`, `CLASS_NAMES`, ...), nên **không được xóa/bỏ qua** các cell phía trên.


## 1. Setup môi trường

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "torch", "torchvision", "scikit-learn", "seaborn", "matplotlib", "tqdm"])
    print("Colab detected: dependencies installed.")
else:
    print("Local environment detected: assuming requirements.txt already installed.")


In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

DATA_DIR = Path("data")
FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]


## 2. Tải dữ liệu & Data Augmentation

Data augmentation (chỉ áp dụng cho tập train): lật ngang, xoay nhẹ, tịnh tiến nhẹ — giúp mô hình tổng quát hóa tốt hơn, giảm overfitting.

In [ ]:
MEAN, STD = (0.2860,), (0.3530,)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

BATCH_SIZE = 128
VAL_SPLIT = 0.1

full_train_aug = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=train_transform)
full_train_eval = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=eval_transform)
test_set = datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=eval_transform)

n_val = int(len(full_train_aug) * VAL_SPLIT)
n_train = len(full_train_aug) - n_val
generator = torch.Generator().manual_seed(SEED)
train_idx, val_idx = random_split(range(len(full_train_aug)), [n_train, n_val], generator=generator)

train_set = Subset(full_train_aug, train_idx.indices)
val_set = Subset(full_train_eval, val_idx.indices)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")


### Xem thử một số ảnh mẫu

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
raw_set = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True)
for ax in axes.flat:
    idx = random.randrange(len(raw_set))
    img, label = raw_set[idx]
    ax.imshow(img, cmap="gray")
    ax.set_title(CLASS_NAMES[label], fontsize=8)
    ax.axis("off")
fig.tight_layout()
plt.show()


## 3. Định nghĩa mô hình

- **MLP**: baseline fully-connected, có BatchNorm + Dropout.
- **CNN**: 2 khối Conv (Conv-BN-ReLU x2 -> MaxPool -> Dropout), sau đó fully-connected có BatchNorm + Dropout.

In [ ]:
class MLP(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class CNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(dropout / 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(dropout / 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


## 4. Vòng lặp huấn luyện / đánh giá

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


def fit(model, train_loader, val_loader, epochs=20, lr=1e-3, weight_decay=1e-4):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    progress = tqdm(range(1, epochs + 1), desc="Epochs")
    for epoch in progress:
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        progress.set_postfix(
            train_loss=f"{train_loss:.4f}", train_acc=f"{train_acc:.4f}",
            val_loss=f"{val_loss:.4f}", val_acc=f"{val_acc:.4f}",
        )
    return history


@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(DEVICE)
        preds = model(images).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


def plot_history(history, title, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title(f"{title} - Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="val")
    axes[1].set_title(f"{title} - Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_confusion(y_true, y_pred, title, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


## Load mô hình đã train sẵn (không train lại ở đây)

Notebook này **chỉ đọc lại** checkpoint đã train sẵn từ Google Drive (thư mục `MyDrive/BTL_AI_checkpoints/`) — checkpoint này do `Fashion_MNIST_CNN.ipynb` tạo ra. Notebook này **không tự train mô hình**.

Khi chạy cell dưới, Colab sẽ hỏi xin quyền truy cập Google Drive — bấm **Allow** để tiếp tục.

**Nếu cell báo lỗi `FileNotFoundError` không tìm thấy checkpoint** — nghĩa là bạn chưa chạy `Fashion_MNIST_CNN.ipynb` (Runtime > Run all) để train và lưu checkpoint lên Drive. Hãy chạy file đó trước, rồi quay lại chạy notebook này.

**Nếu dòng in ra bên dưới là `checkpoints` (không có `/content/drive/...`)** — nghĩa là Drive chưa được mount thật sự (thường do bạn restart runtime rồi chạy nhảy cóc, bỏ qua cell này). Cách khắc phục: **Runtime > Restart session**, sau đó **Run all** lại từ đầu.


In [ ]:
import json

# Tự kiểm tra trực tiếp tại đây (không dựa vào biến IN_COLAB tính ở cell trước đó),
# để dù bạn restart runtime rồi chạy nhảy cóc từ giữa notebook, cell này vẫn luôn
# mount đúng Google Drive thay vì rơi nhầm vào đọc local.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_CKPT_DIR = Path("/content/drive/MyDrive/BTL_AI_checkpoints")
except ImportError:
    DRIVE_CKPT_DIR = Path("checkpoints")  # không chạy trên Colab: đọc local

print("Thư mục đọc checkpoint:", DRIVE_CKPT_DIR.resolve())


def load_checkpoint(model_class, name, dropout=0.3):
    """Load model + history đã train sẵn (do Fashion_MNIST_CNN.ipynb lưu lên Drive).
    KHÔNG train ở đây -- báo lỗi rõ ràng nếu chưa có checkpoint."""
    ckpt_path = DRIVE_CKPT_DIR / f"{name}.pt"
    history_path = DRIVE_CKPT_DIR / f"{name}_history.json"

    if not (ckpt_path.exists() and history_path.exists()):
        raise FileNotFoundError(
            f"Không tìm thấy checkpoint '{name}' tại {DRIVE_CKPT_DIR}. "
            "Hãy chạy Fashion_MNIST_CNN.ipynb (Runtime > Run all) trước để train và lưu checkpoint lên Drive."
        )

    model = model_class(dropout=dropout)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.to(DEVICE)
    with open(history_path) as f:
        history = json.load(f)
    print(f"[{name}] Đã load checkpoint có sẵn trên Drive.")
    return model, history


## 5. Load mô hình MLP đã train sẵn từ Drive


In [ ]:
EPOCHS = 20  # dùng cho phần thực nghiệm nâng cao bên dưới (ablation study)

mlp_model, mlp_history = load_checkpoint(MLP, "mlp")
plot_history(mlp_history, "MLP", save_path=FIGURE_DIR / "mlp_history.png")


## 6. Load mô hình CNN đã train sẵn từ Drive


In [ ]:
cnn_model, cnn_history = load_checkpoint(CNN, "cnn")
plot_history(cnn_history, "CNN", save_path=FIGURE_DIR / "cnn_history.png")


## 7. Đánh giá trên tập test: Accuracy, Precision, Recall, F1-score, Confusion Matrix

In [ ]:
mlp_pred, mlp_true = predict(mlp_model, test_loader)
print("=== MLP classification report ===")
mlp_report = classification_report(mlp_true, mlp_pred, target_names=CLASS_NAMES, digits=4, output_dict=True)
print(classification_report(mlp_true, mlp_pred, target_names=CLASS_NAMES, digits=4))
plot_confusion(mlp_true, mlp_pred, "MLP Confusion Matrix", save_path=FIGURE_DIR / "mlp_confusion_matrix.png")


In [ ]:
cnn_pred, cnn_true = predict(cnn_model, test_loader)
print("=== CNN classification report ===")
cnn_report = classification_report(cnn_true, cnn_pred, target_names=CLASS_NAMES, digits=4, output_dict=True)
print(classification_report(cnn_true, cnn_pred, target_names=CLASS_NAMES, digits=4))
plot_confusion(cnn_true, cnn_pred, "CNN Confusion Matrix", save_path=FIGURE_DIR / "cnn_confusion_matrix.png")


## 8. So sánh CNN vs MLP

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["MLP", "CNN"],
    "Accuracy": [mlp_report["accuracy"], cnn_report["accuracy"]],
    "Precision (macro)": [mlp_report["macro avg"]["precision"], cnn_report["macro avg"]["precision"]],
    "Recall (macro)": [mlp_report["macro avg"]["recall"], cnn_report["macro avg"]["recall"]],
    "F1-score (macro)": [mlp_report["macro avg"]["f1-score"], cnn_report["macro avg"]["f1-score"]],
    "Params": [sum(p.numel() for p in mlp_model.parameters()), sum(p.numel() for p in cnn_model.parameters())],
})
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mlp_history["val_acc"], label="MLP val_acc")
ax.plot(cnn_history["val_acc"], label="CNN val_acc")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Accuracy")
ax.set_title("MLP vs CNN - Validation Accuracy qua các epoch")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mlp_vs_cnn_val_acc.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Lưu mô hình (tuỳ chọn)

Việc lưu checkpoint chính thức lên Google Drive được thực hiện trong `Fashion_MNIST_CNN.ipynb`. Cell dưới đây chỉ lưu thêm 1 bản local trong phiên Colab hiện tại (sẽ mất khi ngắt kết nối) — không bắt buộc chạy.


In [ ]:
CKPT_DIR = Path("checkpoints")
CKPT_DIR.mkdir(exist_ok=True)
torch.save(mlp_model.state_dict(), CKPT_DIR / "mlp.pt")
torch.save(cnn_model.state_dict(), CKPT_DIR / "cnn.pt")
print("Saved checkpoints to", CKPT_DIR.resolve())


## 10. Kết luận

Điền nhận xét dựa trên bảng so sánh và các biểu đồ ở trên, ví dụ:
- CNN có Accuracy/F1-score cao hơn MLP hay không, chênh lệch bao nhiêu?
- Data Augmentation, Dropout, Batch Normalization ảnh hưởng thế nào đến khoảng cách giữa train và val accuracy (overfitting)?
- Các lớp nào (class) bị nhầm lẫn nhiều nhất qua confusion matrix (ví dụ Shirt vs T-shirt/top vs Pullover vs Coat)?


---
# Phần bổ sung: Thực nghiệm nâng cao cho báo cáo

4 mục dưới đây tương ứng với checklist trong `REPORT_GUIDE.md`, giúp báo cáo có chiều sâu hơn thay vì chỉ dừng ở "train xong, so sánh accuracy".


## 11. Ablation Study — Từng kỹ thuật cải thiện có thực sự hiệu quả?

CNN ở mục 6 được train với **đầy đủ 3 kỹ thuật**: Data Augmentation, Dropout, Batch Normalization. Nhưng làm sao biết chắc từng kỹ thuật này thực sự có ích?

Cách kiểm chứng (gọi là **ablation study** — "cắt bỏ từng phần rồi xem điều gì xảy ra"): train lại CNN 3 lần nữa, mỗi lần **bỏ đúng 1 kỹ thuật**, rồi so sánh với bản đầy đủ qua 2 con số:

- **Test Accuracy** — độ chính xác trên dữ liệu chưa từng thấy (test set). Càng cao càng tốt.
- **Overfitting Gap** = Accuracy trên tập train − Accuracy trên tập validation. Nếu số này lớn, nghĩa là mô hình "học thuộc lòng" tập train rất giỏi nhưng gặp dữ liệu mới (val) thì kém hơn hẳn — đây gọi là hiện tượng **overfitting** (học vẹt, không tổng quát hóa được). Gap càng thấp càng tốt.

4 cấu hình sẽ so sánh:
1. **CNN đầy đủ** — dùng lại `cnn_model`/`cnn_history` đã train ở mục 6 (không train lại, đỡ tốn thời gian)
2. **CNN không Data Augmentation** — train bằng ảnh gốc, không xoay/lật/tịnh tiến
3. **CNN không Dropout** — bỏ hẳn lớp Dropout (`dropout=0.0`)
4. **CNN không Batch Normalization** — bỏ hẳn lớp BatchNorm

Lưu ý: mỗi lần train tốn thời gian tương đương lúc train CNN ở mục 6 (dùng cùng `EPOCHS`), nên tổng cộng mục này sẽ chạy thêm ~3 lần thời gian đó. Nếu muốn nhanh hơn, có thể giảm biến `ABLATION_EPOCHS` bên dưới.


In [ ]:
# Loader huấn luyện KHÔNG augmentation (dùng ảnh gốc, không xoay/lật/tịnh tiến)
# Dùng lại đúng các ảnh train (train_idx) nhưng lấy từ full_train_eval (transform không augment)
train_set_noaug = Subset(full_train_eval, train_idx.indices)
train_loader_noaug = DataLoader(train_set_noaug, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
print(f"Train (không augment): {len(train_set_noaug)} ảnh")

ABLATION_EPOCHS = EPOCHS  # đổi thành số nhỏ hơn (vd 10) nếu muốn chạy nhanh hơn


In [ ]:
class CNNNoBatchNorm(nn.Module):
    """Giống CNN ở mục 3, nhưng bỏ toàn bộ lớp BatchNorm — dùng cho ablation study."""

    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(dropout / 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(dropout / 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [ ]:
print("Training: CNN không Augmentation...")
cnn_noaug = CNN(dropout=0.3)
history_noaug = fit(cnn_noaug, train_loader_noaug, val_loader, epochs=ABLATION_EPOCHS)


In [ ]:
print("Training: CNN không Dropout...")
cnn_nodrop = CNN(dropout=0.0)
history_nodrop = fit(cnn_nodrop, train_loader, val_loader, epochs=ABLATION_EPOCHS)


In [ ]:
print("Training: CNN không BatchNorm...")
cnn_nobn = CNNNoBatchNorm(dropout=0.3)
history_nobn = fit(cnn_nobn, train_loader, val_loader, epochs=ABLATION_EPOCHS)


In [ ]:
def final_gap(history):
    return history["train_acc"][-1] - history["val_acc"][-1]

ablation_variants = {
    "CNN đầy đủ": (cnn_model, cnn_history),
    "Không Augmentation": (cnn_noaug, history_noaug),
    "Không Dropout": (cnn_nodrop, history_nodrop),
    "Không BatchNorm": (cnn_nobn, history_nobn),
}

ablation_rows = []
for name, (model, history) in ablation_variants.items():
    preds, trues = predict(model, test_loader)
    test_acc = (preds == trues).mean()
    ablation_rows.append({
        "Cấu hình": name,
        "Test Accuracy": round(float(test_acc), 4),
        "Train Acc (cuối)": round(history["train_acc"][-1], 4),
        "Val Acc (cuối)": round(history["val_acc"][-1], 4),
        "Overfitting Gap": round(final_gap(history), 4),
    })

ablation_df = pd.DataFrame(ablation_rows)
ablation_df


**Cách đọc bảng trên:**
- Cột `Test Accuracy` càng cao càng tốt — đo trên dữ liệu mô hình chưa từng thấy.
- Cột `Overfitting Gap` càng **thấp** càng tốt. Gap cao nghĩa là mô hình học thuộc lòng tập train, gặp dữ liệu mới thì kém hẳn.
- Nếu dòng "Không Augmentation" hoặc "Không Dropout" có Gap cao hơn hẳn "CNN đầy đủ" → đây là bằng chứng số cho thấy kỹ thuật đó thực sự giúp giảm overfitting, đúng như lý thuyết đã nêu trong đề tài.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(ablation_df["Cấu hình"], ablation_df["Test Accuracy"], color="#4C72B0")
axes[0].set_title("Test Accuracy theo từng cấu hình")
axes[0].set_ylabel("Accuracy")
axes[0].tick_params(axis="x", rotation=20)
for i, v in enumerate(ablation_df["Test Accuracy"]):
    axes[0].text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)

axes[1].bar(ablation_df["Cấu hình"], ablation_df["Overfitting Gap"], color="#C44E52")
axes[1].set_title("Overfitting Gap (Train Acc − Val Acc)")
axes[1].set_ylabel("Gap")
axes[1].tick_params(axis="x", rotation=20)
for i, v in enumerate(ablation_df["Overfitting Gap"]):
    axes[1].text(i, v + 0.002, f"{v:.3f}", ha="center", fontsize=9)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "ablation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


**Đọc biểu đồ:**
- **Biểu đồ trái:** cột càng cao = mô hình đoán đúng nhiều hơn trên tập test. So sánh cột "CNN đầy đủ" với 3 cột còn lại để biết thiếu kỹ thuật nào ảnh hưởng đến độ chính xác nhiều nhất.
- **Biểu đồ phải (quan trọng nhất của mục này):** cột càng cao = overfitting càng nặng. Đây chính là "bằng chứng số" cho việc Dropout/Augmentation/BatchNorm giúp mô hình tổng quát hóa tốt hơn.
- Khi viết báo cáo, hãy ghi lại: cấu hình nào có Gap cao nhất, và giải thích tại sao (ví dụ: bỏ Augmentation khiến CNN nhìn đi nhìn lại đúng những ảnh cũ nhiều lần → dễ học thuộc lòng → Gap cao hơn).


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, (_, history) in ablation_variants.items():
    ax.plot(history["val_acc"], label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Accuracy")
ax.set_title("So sánh tốc độ học (Validation Accuracy) giữa các cấu hình")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "ablation_val_acc_curves.png", dpi=150, bbox_inches="tight")
plt.show()


**Đọc biểu đồ:** mỗi đường là 1 cấu hình; trục ngang là số vòng học (epoch), trục dọc là độ chính xác trên tập validation. Đường nào **nằm trên cao** và **ít dao động lên xuống** là cấu hình học tốt và ổn định hơn. Nếu đường "Không BatchNorm" dao động mạnh hoặc nằm thấp hơn hẳn các đường khác, đó là bằng chứng cho thấy BatchNorm giúp quá trình học ổn định hơn qua từng epoch.


## 12. Phân tích lỗi sai (Error Analysis)

Phần này giúp hiểu **tại sao** CNN sai, chứ không chỉ biết nó sai bao nhiêu %. Đây là phần giám khảo thường đánh giá cao vì cho thấy người làm hiểu bản chất, không chỉ chạy code lấy số.


In [ ]:
cnn_pred_final, cnn_true_final = predict(cnn_model, test_loader)
wrong_idx = np.where(cnn_pred_final != cnn_true_final)[0]
print(f"CNN đoán sai {len(wrong_idx)} / {len(cnn_true_final)} ảnh (tỉ lệ sai: {len(wrong_idx) / len(cnn_true_final):.2%})")


In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(15, 7))
sample_wrong = np.random.choice(wrong_idx, size=18, replace=False)
for ax, idx in zip(axes.flat, sample_wrong):
    img, _ = test_set[idx]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(
        f"Thật: {CLASS_NAMES[cnn_true_final[idx]]}\nĐoán: {CLASS_NAMES[cnn_pred_final[idx]]}",
        fontsize=8, color="crimson",
    )
    ax.axis("off")
fig.suptitle("18 ảnh CNN đoán SAI (chọn ngẫu nhiên)", fontsize=13)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "error_analysis_samples.png", dpi=150, bbox_inches="tight")
plt.show()


**Đọc hình trên:** mỗi ô là 1 ảnh CNN đoán sai, dòng chữ đỏ cho biết nhãn thật (Thật) và nhãn CNN đoán nhầm thành (Đoán). Hãy tự nhìn bằng mắt: nếu bạn cũng thấy khó phân biệt 2 nhãn đó (ví dụ ảnh mờ, góc chụp lạ, vật thể bị cắt góc), thì đây là lỗi "hợp lý" — chứng tỏ CNN sai vì dữ liệu khó, không phải vì mô hình kém. Ngược lại nếu ảnh rất rõ ràng mà CNN vẫn đoán sai, đáng để ghi chú là hạn chế của mô hình.


In [ ]:
cm_final = confusion_matrix(cnn_true_final, cnn_pred_final)
pairs = []
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        if i != j and cm_final[i, j] > 0:
            pairs.append((f"{CLASS_NAMES[i]} → {CLASS_NAMES[j]}", cm_final[i, j]))
pairs.sort(key=lambda x: x[1], reverse=True)
top_pairs = pairs[:10]

fig, ax = plt.subplots(figsize=(9, 5))
labels = [p[0] for p in top_pairs]
counts = [p[1] for p in top_pairs]
ax.barh(labels[::-1], counts[::-1], color="#DD8452")
ax.set_xlabel("Số lần nhầm lẫn")
ax.set_title("Top 10 cặp nhãn hay bị CNN nhầm lẫn nhất (Thật → Đoán nhầm thành)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "top_confused_pairs.png", dpi=150, bbox_inches="tight")
plt.show()


**Đọc biểu đồ:** mỗi thanh ngang là 1 cặp "nhãn thật → nhãn bị đoán nhầm thành", thanh càng dài càng bị nhầm nhiều lần. Ví dụ nếu thanh dài nhất là "Shirt → T-shirt/top", nghĩa là CNN hay nhầm áo sơ mi thành áo thun — điều này hợp lý vì 2 loại trang phục này khá giống nhau khi nhìn ở ảnh xám nhỏ 28x28 pixel. Đưa top 2-3 cặp nhầm lẫn nhiều nhất vào báo cáo, kèm giải thích vì sao chúng dễ gây nhầm (hình dáng giống nhau, thiếu chi tiết phân biệt như tay áo dài/ngắn...).


## 13. So sánh chi phí tính toán (Model Complexity)

CNN chính xác hơn MLP, nhưng "đắt" hơn bao nhiêu? Phần này đo 2 thứ:
- **Số tham số** (parameters) — mô hình càng nhiều tham số càng "nặng", tốn nhiều bộ nhớ hơn.
- **Thời gian train mỗi epoch** — mô hình càng lâu, càng tốn tài nguyên/thời gian huấn luyện.

Đây là phần thể hiện **trade-off** (đánh đổi): đổi lấy độ chính xác cao hơn, ta phải trả giá bằng chi phí tính toán cao hơn — nên nhắc tới trong phần kết luận báo cáo.


In [ ]:
import time


def measure_epoch_time(model, loader):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    start = time.time()
    train_one_epoch(model, loader, optimizer, criterion)
    return time.time() - start


mlp_time = measure_epoch_time(MLP(dropout=0.3), train_loader)
cnn_time = measure_epoch_time(CNN(dropout=0.3), train_loader)

cost_df = pd.DataFrame({
    "Model": ["MLP", "CNN"],
    "Số tham số": [
        sum(p.numel() for p in mlp_model.parameters()),
        sum(p.numel() for p in cnn_model.parameters()),
    ],
    "Thời gian / epoch (giây)": [round(mlp_time, 2), round(cnn_time, 2)],
    "Test Accuracy": [round(mlp_report["accuracy"], 4), round(cnn_report["accuracy"], 4)],
})
cost_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(cost_df["Model"], cost_df["Số tham số"], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Số tham số")

axes[1].bar(cost_df["Model"], cost_df["Thời gian / epoch (giây)"], color=["#4C72B0", "#DD8452"])
axes[1].set_title("Thời gian train / epoch (giây)")

axes[2].bar(cost_df["Model"], cost_df["Test Accuracy"], color=["#4C72B0", "#DD8452"])
axes[2].set_title("Test Accuracy")

fig.tight_layout()
fig.savefig(FIGURE_DIR / "cost_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


**Đọc biểu đồ:** 3 biểu đồ cột đặt cạnh nhau để so sánh trực tiếp MLP (xanh) và CNN (cam) trên 3 tiêu chí. Cách đọc: lấy tỉ lệ chênh lệch — ví dụ "CNN có Accuracy cao hơn MLP X điểm %, nhưng đổi lại có số tham số gấp Y lần và thời gian train gấp Z lần". Đây là câu kết luận thường thấy trong các báo cáo so sánh mô hình: mô hình phức tạp hơn thường chính xác hơn nhưng tốn chi phí hơn, cần cân nhắc tùy bài toán thực tế (ví dụ chạy trên điện thoại thì ưu tiên mô hình nhẹ hơn dù kém chính xác hơn 1 chút).


## 14. Trực quan hóa: CNN "nhìn thấy" gì trong ảnh?

Đây là phần giải thích trực quan **tại sao** CNN học tốt hơn MLP, thay vì chỉ dựa vào con số. Ta sẽ lấy 1 ảnh test, cho đi qua lớp Conv **đầu tiên** của CNN, và xem kết quả (gọi là **feature map**) trông như thế nào.


In [ ]:
sample_img, sample_label = test_set[0]
input_tensor = sample_img.unsqueeze(0).to(DEVICE)

cnn_model.eval()
with torch.no_grad():
    fmap1 = cnn_model.features[0](input_tensor)  # đầu ra ngay sau lớp Conv2d đầu tiên

fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(sample_img.squeeze(), cmap="gray")
ax.set_title(f"Ảnh gốc: {CLASS_NAMES[sample_label]}")
ax.axis("off")
plt.show()

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(fmap1[0, i].cpu(), cmap="viridis")
    ax.axis("off")
fig.suptitle("32 feature map sau lớp Conv đầu tiên", fontsize=13)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "feature_maps_conv1.png", dpi=150, bbox_inches="tight")
plt.show()


**Đọc hình trên:** mỗi ô nhỏ trong lưới 4×8 là kết quả khi ảnh gốc đi qua 1 trong 32 "bộ lọc" (filter) của lớp Conv đầu tiên. Một số bộ lọc bắt được **viền/cạnh** của vật thể (giống hiệu ứng "dò biên" trong Photoshop), số khác bắt được **vùng sáng/tối** hoặc **hoa văn**. Đây chính là lý do CNN "nhìn" ảnh tốt hơn MLP: CNN **tự học ra** các bộ lọc phát hiện đặc trưng hình học (cạnh, góc, kết cấu vải), trong khi MLP chỉ xử lý từng pixel riêng lẻ như những con số độc lập, không hiểu được pixel nào nằm cạnh pixel nào trong không gian ảnh.


## Kết luận tổng hợp (bổ sung)

Dựa trên các thực nghiệm ở mục 11-14, hãy điền nhận xét vào báo cáo, ví dụ theo khung sau:

1. **So sánh MLP vs CNN** (mục 7-8): CNN đạt Accuracy ___%, cao hơn MLP ___ điểm %. CNN có nhiều tham số hơn MLP ___ lần, thời gian train mỗi epoch lâu hơn ___ lần (mục 13).
2. **Ablation study** (mục 11): kỹ thuật ảnh hưởng nhiều nhất đến overfitting là ___ (dựa vào cột Overfitting Gap). Kỹ thuật ảnh hưởng nhiều nhất đến Accuracy là ___.
3. **Lỗi sai thường gặp** (mục 12): CNN hay nhầm lẫn nhất giữa cặp nhãn ___ và ___, lý do có thể là ___.
4. **Vì sao CNN hiệu quả hơn MLP** (mục 14): CNN học được các bộ lọc phát hiện cạnh/hoa văn — phù hợp với đặc điểm ảnh có tính không gian, còn MLP không tận dụng được điều này.
5. **Hạn chế & hướng cải thiện**: có thể thử kiến trúc sâu hơn (thêm khối Conv), thử kỹ thuật augmentation khác (CutMix, MixUp), hoặc tăng số epoch/tinh chỉnh learning rate.
